In [ ]:
# notebook cell — validate the 3D STM against finite differences
import numpy as np
import sys
sys.path.insert(0, '../src')
from cr3bp_3d import (cr3bp_rhs_3d, integrate_with_stm_3d,
                      jacobi_constant_3d, MU_EARTH_MOON)
from scipy.integrate import solve_ivp

mu = MU_EARTH_MOON

# A reference 3D initial condition (with nonzero z and vz, to exercise all terms)
x0 = np.array([0.82, 0.0, 0.1, 0.0, 0.15, 0.05])  # [x, y, z, vx, vy, vz]
t_final = 1.5

# Integrate reference trajectory WITH the STM
sol_ref = integrate_with_stm_3d(x0, (0, t_final), mu, t_eval=[t_final])
ref_final = sol_ref.y[:6, -1]
Phi = sol_ref.y[6:, -1].reshape(6, 6)

print("STM Phi at t_final (6x6):")
print(np.array2string(Phi, precision=3, suppress_small=True))

# Test the STM against finite differences for ALL 6 perturbation directions
epsilon = 1e-6
print(f"\n{'component':>10} | {'max error':>12}")
print("-" * 28)
max_overall = 0.0
for j in range(6):
    dx0 = np.zeros(6)
    dx0[j] = epsilon
    sol_pert = solve_ivp(cr3bp_rhs_3d, (0, t_final), x0 + dx0, args=(mu,),
                         method='DOP853', rtol=1e-12, atol=1e-12, t_eval=[t_final])
    pert_final = sol_pert.y[:, -1]
    dx_actual = pert_final - ref_final
    dx_predicted = Phi @ dx0
    max_err = np.max(np.abs(dx_actual - dx_predicted))
    max_overall = max(max_overall, max_err)
    print(f"{j:>10} | {max_err:>12.2e}")

print(f"\nOverall max error: {max_overall:.2e}")
print("(Should be ~1e-9 to 1e-11 if the 6x6 Jacobian is correct)")

STM Phi at t_final (6x6):
[[ 10.21   -2.833  -1.146   3.116   2.235  -0.262]
 [ -8.204   1.696   1.412  -3.103  -1.141   0.391]
 [ -0.937   0.252  -0.519  -0.285  -0.028   0.43 ]
 [ 15.632  -5.399  -1.36    4.427   3.598  -0.31 ]
 [-13.584   3.17    1.713  -4.191  -3.279   0.424]
 [ -0.356   0.147  -1.14   -0.14   -0.183  -0.822]]

 component |    max error
----------------------------
         0 |     3.36e-10
         1 |     2.07e-11
         2 |     1.63e-11
         3 |     4.27e-11
         4 |     1.22e-11
         5 |     1.42e-12

Overall max error: 3.36e-10
(Should be ~1e-9 to 1e-11 if the 6x6 Jacobian is correct)


In [8]:
from cr3bp_3d import cr3bp_jacobian_3d
print(cr3bp_jacobian_3d(x0, mu))

[[ 0.          0.          0.          1.          0.          0.        ]
 [ 0.          0.          0.          0.          1.          0.        ]
 [ 0.          0.          0.          0.          0.          1.        ]
 [ 6.26192932  0.         -1.5527329   0.          2.          0.        ]
 [ 0.         -2.30694696  0.         -2.          0.          0.        ]
 [-1.5527329   0.         -1.95498236  0.          0.          0.        ]]


In [6]:
J = np.block([[np.zeros((3,3)), np.eye(3)],[-np.eye(3), np.zeros((3,3))]])
print(np.max(np.abs(Phi.T @ J @ Phi - J)))   # match J ordering to your state vector

13.85701214889069
